# Week 13 Discussion: Recommender Systems Analysis
## Reddit's Home Feed and Community Recommendation Engine
**Student:** Daniel Foulen  
**Class:** IS 362  
**Date:** 5/3/2026

## Overview
Reddit is a social platform built around communities (subreddits) organized by topic or interest. Its recommendation engine surfaces content in two ways: ranking posts within the home feed, and suggesting new communities to users based on behavior.

Unlike the UCI Mushroom dataset from HW8 where features were clean, categorical, and well-defined, Reddit's recommendation problem operates on implicit behavioral signals (upvotes, dwell time, click patterns, community overlap). The signal is messy, and what the org-created algorithm optimizes for is not always what users would say they want.

## The Central Question
Reddit's recommendation engine uses one mechanism to produce two very different outcomes. The same system that pulls a new hiker deeper into outdoor communities can pull a politically curious user into increasingly narrow ideological spaces. This analysis examines both pathways to understand what the mechanism is actually optimizing for.

## Why This System Is Worth Analyzing
Recommender systems are usually evaluated on engagement metrics (clicks, time on site, return visits). Reddit's system performs well by those measures, but engagement is not the same as user benefit, and the gap between the two is where the most interesting design questions live. Understanding that gap is useful in any domain where recommendations shape behavior over time, not just social media, and explores what sort of trust we should give or take back from these systems.

## Section 1: Scenario Design (Reddit as an Organization)

Scenario Design asks three questions: who are the target users, what are their key goals, and how does the system help accomplish those goals. Here I apply it first from Reddit's perspective as a business, then from the user's perspective.

**Who are Reddit's target users (as a business)?**  
Reddit's commercial customers are advertisers. Its product is user attention. As of Q4 2024, Reddit reported 101.7 million daily active unique users across more than 100,000 active communities (Reddit 10-K Annual Report, 2024:
https://www.sec.gov/Archives/edgar/data/1713445/000171344525000096/redditannualreport2024.pdf).

**What are Reddit's key goals?**  
- Maximize time on platform
- Increase return visit frequency
- Grow the number of active communities (more surface area for ad targeting)
- Keep new users from churning before they find communities worth staying for

**How does the recommendation system serve those goals?**  
The home feed keeps users scrolling by surfacing content that matches demonstrated preferences. Suggested communities extend the session by giving users new places to go when their current feed feels stale. Both functions serve the same business objective: sustained engagement.

Reddit's 10-K ties user growth and engagement directly to advertising revenue, framing recommendation quality as a driver of business outcomes. The system is designed with commercial optimization in mind, even when it presents itself as community-driven discovery.

## Section 2: Scenario Design (The User)

The same three questions applied to users and the contrast between the two scenario designs explore the effects of this infrastructure.

**Who are Reddit's users?**  
Reddit's user base is broad, but two personas are useful for this analysis.

*Persona A ("Oh wow this is cool")*  
A user who recently developed an interest (a new hobby, a health diagnosis, a life transition). They are actively looking for community and relevant information. They have sparse behavioral history on the platform.

*Persona B ("(Insert Politician) is a loser")*  
A user with a dense behavioral history concentrated in a specific topic area, particularly a politically or ideologically charged one. They have strong, consistent signals that the algorithm has been optimizing on for months or years.

**What are users' key goals?**  
- Find content and communities relevant to their genuine interests
- Discover things they did not know they wanted (serendipity)
- Feel like the platform is working for them, not on them

**How does the system serve or fail those goals?**  
For Persona A, the system works reasonably well early on. Sparse history means the algorithm draws on community-level signals and surfaces a range of related subreddits. Discovery is broad before it narrows.

For Persona B, the system has already narrowed. Dense behavioral history produces high-confidence recommendations that reinforce existing patterns. The algorithm is doing exactly what it was designed to do, and that is the problem worth examining.

## Section 3: Reverse Engineering the System

Reddit has never published a complete technical specification of its recommendation system, but significant detail is reconstructable from its open-sourced code (2009), public research, and observable interface behavior.

### Post Ranking (The Hot Algorithm)
Reddit's hot sort ranks posts using a formula that combines vote score and submission time. The key properties (Evan Miller, 2015:
https://www.evanmiller.org/deriving-the-reddit-formula.html):

- **Logarithmic vote scaling**: The first 10 upvotes carry the same weight as the next 100,
  which carry the same weight as the next 1000. Early votes matter disproportionately.
- **Time component**: Posts earn a bonus based on seconds elapsed since Reddit's founding
  (December 8, 2005), divided by 45,000. Newer posts get a natural boost, but decay is
  competitive rather than forced. A post does not lose score over time, it gets
  outpaced by newer content.
- **Downvote asymmetry**: Posts that accumulate both upvotes and downvotes rank lower
  than posts that accumulate only upvotes. This structurally disadvantages
  challenging or dissenting content.

### Comment Ranking (The Wilson Score Interval)
Comments use a different system. 
The Wilson score interval, applied to Reddit's comment ranking via Evan Miller's statistical writeup, treats votes as a statistical sample (Evan Miller, "How Not to Sort by Average Rating": https://www.evanmiller.org/how-not-to-sort-by-average-rating.html). 
A comment with 10 upvotes and 1 downvote can outrank one with 40 upvotes and 20 downvotes because statistical confidence in its quality is higher. 
Submission time is irrelevant here, which is appropriate for comments but would be wrong for time-sensitive posts.

### Home Feed Personalization
Based on observable interface behavior and Reddit's public disclosures in its 10-K, the home feed blends content from three sources:
1. Posts from subscribed communities, weighted by interaction frequency
2. Recommended communities inferred from behavioral overlap with similar users
3. High-performing posts predicted to match the user's taste profile

Signals that appear to drive ranking include upvotes, comments, dwell time, and hide/report actions. 
Reddit has publicly acknowledged reducing the weight of downvotes in feed ranking, citing manipulation concerns. 
The tradeoff is that this also reduces the platform's ability to suppress low-quality or harmful content through community signal.

## Section 4: The Dual Mechanism

The home feed and community recommendation system operate on a single underlying logic:
find what a user has engaged with, find other users with similar engagement patterns, surface content and communities that overlap. 
This is collaborative filtering applied to implicit behavioral data.

The mechanism is content-agnostic. 
It does not distinguish between a user deepening their interest in trail running and a user deepening their exposure to conspiratorial political content. 
Both produce the same kind of signal.

### Pathway 1 (Hobby Exploration)
A user searches for beginner hiking advice. They join r/hiking. 
They upvote gear recommendations and trip reports. The algorithm observes that users with similar patterns also engage with r/ultralight, r/backpacking, r/CampingandHiking. These communities get surfaced. 

The user's feed expands into a coherent, enriching topic cluster. 

Discovery is working as intended.

### Pathway 2 (Ideological Narrowing)
A user engages with politically charged content in a subreddit. They upvote, comment, return. 
The algorithm observes behavioral overlap with users in adjacent communities that share tone and framing, even if ostensibly covering different topics. 
Those communities get surfaced. The user's feed narrows into a reinforcing cluster. The algorithm has not done anything different from Pathway 1. 
It has applied the same logic to a domain where narrowing has different consequences.

### What the Research Actually Shows
It is worth being precise here... 

Research does not uniformly support the "algorithm causes radicalization" narrative.

A study examining Reddit's r/The_Donald found the upvoting algorithm amplified extreme discourse within that community. 
Posts containing extreme content were disproportionately surfaced compared to random samples from the same subreddit
(Gaudette et al., 2020, as cited in Whittaker et al., 2021:
https://policyreview.info/articles/analysis/recommender-systems-and-amplification-extremist-content).

However, a comparative study across YouTube, Reddit, and Gab found evidence of algorithmic amplification on YouTube specifically, but not on Reddit...
Suggesting that on Reddit, user behavior may drive content more than the recommendation system does (Whittaker et al., 2021:
https://policyreview.info/articles/analysis/recommender-systems-and-amplification-extremist-content).

A PNAS study with nearly 9,000 participants found that manipulating algorithmic recommendations to create filter bubble conditions had limited polarization effects, noting that users' preexisting attitudes and choices matter as much as the algorithm
(Liu et al., 2025: https://www.pnas.org/doi/10.1073/pnas.2318127122).

It is probably not that Reddit's algorithm causes radicalization, but rather that the algorithm is indifferent to the nature of what it amplifies. 
In domains where amplification has downstream social consequences, indifference is itself a design choice worth examining.

## Section 5: Recommendations

The following recommendations are specific and mechanistic. "Be more ethical" is not a recommendation. What follows are design changes that would alter the system's behavior in targeted ways.

**1. Restore downvote signal in feed ranking**  
Reddit reduced downvote weight to prevent manipulation, but downvotes are also the primary community mechanism for suppressing low-quality content. A more targeted fix would weight downvotes differently based on source. Downvotes from accounts with longer tenure and broader community engagement should carry more signal than coordinated downvotes from new or single-community accounts.

**2. Add a diversity floor to community recommendations**  
The current system surfaces communities with the highest behavioral overlap. A diversity floor would require a percentage of recommended communities to come from outside the user's inferred interest cluster. The New York Times recommendation engine implemented a similar idea through a back-off approach that assumed users only 90% prefer what they clicked, leaving room for serendipitous recommendations
(Spangher, 2015:
http://open.blogs.nytimes.com/2015/08/11/building-the-next-new-york-times-recommendation-engine/). Reddit could apply the same principle at the community level.

**3. Surface recommendation transparency**  
The interface provides minimal explanation of why a specific post or community was surfaced. A visible "why am I seeing this" feature would give users the ability to correct the algorithm's model of them rather than passively receiving its outputs. This is a feature that other major platforms have implemented in response to user and regulatory pressure.

**4. Distinguish engagement from satisfaction**  
The current system treats all engagement as positive signal. A user who spends ten minutes rage-reading a post generates the same dwell-time signal as a user who found it genuinely useful. Incorporating explicit satisfaction signals (saved posts, return engagement with recommended content) as distinct from raw engagement would let the algorithm optimize for something closer to user benefit rather than just user attention.

## Conclusion

Reddit's recommendation engine is technically coherent and commercially effective.
The hot algorithm is well-designed for its purpose. The personalization system does
what collaborative filtering is supposed to do.

The design questions worth asking are not about whether the system works, but about
what it is working toward. Engagement and benefit are not the same objective, and
the current system was built to optimize the former. The recommendations above are
attempts to close that gap without requiring the system to make editorial judgments
it is not equipped to make.